# Coresearcher (Ollama powered)

This demo shows how to use an AstroFlow-aware coresearcher to help code data projects.

This particular version allows the use of open-source models via `ollama`, which will need to be installed.


In [ ]:
from research_assistant import (
    MemoryStore,
    ResearchAssistant,
    Retriever,
    build_messages,
    start_new_memory,
)


Set up the research assistant. 

In [11]:

# see https://ollama.com/search
model = "llama3.2"
model = "granite4.1:3b"
memory_dir = 'memory'
ra = ResearchAssistant(model=model, memory_dir=memory_dir)


Ask your first question!

In [12]:
ra.ask("What's a random idea for a figure from Gaia DR3 data that I could make in Python using PySpark?")

Here's an idea for a figure: "Distribution of Gaia DR3 Star Parallaxes".

**Idea:**

Plot a histogram of the parallax values of stars in Gaia DR3 to visualize the distribution of measurement uncertainties.

**Python Code:**

```python
import spark
from astroflow_spark_gaia import gaia_data

# Create a SparkSession
spark = gaia_spark_gaia(spark)
df_gaia = spark.sql("SELECT parallax FROM gaia_source_radial")

# Filter out NaN values
df_gaia = df_gaia.na.drop()

# Plot histogram
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(df_gaia.parallax, bins=50, color='steelblue', edgecolor='black')
plt.xlabel('Parallax (mas)')
plt.ylabel('Number of Stars')
plt.title('Distribution of Gaia DR3 Star Parallaxes')
plt.axvline(0.5, color='red', linestyle='dashed', linewidth=1)
plt.annotate('Typical Uncertainty', (0.2, 2000000), textcoords="offset points", xytext=(0,10),
              bbox=dict(boxstyle="round", facecolor="white", alpha=0.5),
              arrowprops=dict(arrowstyle

'Here\'s an idea for a figure: "Distribution of Gaia DR3 Star Parallaxes".\n\n**Idea:**\n\nPlot a histogram of the parallax values of stars in Gaia DR3 to visualize the distribution of measurement uncertainties.\n\n**Python Code:**\n\n```python\nimport spark\nfrom astroflow_spark_gaia import gaia_data\n\n# Create a SparkSession\nspark = gaia_spark_gaia(spark)\ndf_gaia = spark.sql("SELECT parallax FROM gaia_source_radial")\n\n# Filter out NaN values\ndf_gaia = df_gaia.na.drop()\n\n# Plot histogram\nimport matplotlib.pyplot as plt\n\nplt.figure(figsize=(10, 6))\nplt.hist(df_gaia.parallax, bins=50, color=\'steelblue\', edgecolor=\'black\')\nplt.xlabel(\'Parallax (mas)\')\nplt.ylabel(\'Number of Stars\')\nplt.title(\'Distribution of Gaia DR3 Star Parallaxes\')\nplt.axvline(0.5, color=\'red\', linestyle=\'dashed\', linewidth=1)\nplt.annotate(\'Typical Uncertainty\', (0.2, 2000000), textcoords="offset points", xytext=(0,10),\n              bbox=dict(boxstyle="round", facecolor="white", alpha

Ollama models are generally weaker -- there are some syntax errors we need to clean up. This is alright! We have access to the conversation history and can feed in new ideas.

In [ ]:
import spark
from astroflow_spark_gaia import gaia_data

# Create a SparkSession
spark = gaia_spark_gaia(spark)
df_gaia = spark.sql(
    """
    SELECT 
        glon, 
        glat, 
        dist_kpc 
    FROM 
        gaia_source_radial
    WHERE 
        astrometric_excess_magnitude > 10 AND 
        astrometric_excess_magnitude < 13
    """
)

# Select stars with parallax >= 0.5 (i.e., within 100 pc)
df_gaia = df_gaia.filter(df_gaia.dist_kpc >= 0.5)

# Convert to 3D coordinates
df_gaia = df_gaia.withColumn('x', df_gaia.glon)
df_gaia = df_gaia.withColumn('y', df_gaia.glat)
df_gaia = df_gaia.withColumn('z', df_gaia.dist_kpc)

# Plot in 3D
import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d as mplot3d

fig = plt.figure(figsize=(10, 10))
ax = mplot3d.Axes3D(fig)

ax.scatter(df_gaia.x, df_gaia.y, df_gaia.z, c='blue')

ax.set_xlabel('Galactic Longitude')
ax.set_ylabel('Galactic Latitude')
ax.set_zlabel('Distance (kpc)')

plt.show()

The assistant is aware of questions you have asked before. So if the software doesn't work, you can input the error and receive corrections.

The conversation log is in the memory directory you set above.